# H2: one leverage row in the context moves the tail

The mechanism is read from source: none of the five packages protects the target
variable, which is standardised by its plain mean and standard deviation, while
features are as a rule put through outlier-resistant transforms. That part is a fact
about the code, not a hypothesis.

What is measured here is the consequence, and how often the trigger occurs in practice.
Tables are recomputed from `results/h2_leverage/`.

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
sys.path.insert(0, "..")
from common import paths, metrics, tables
pd.set_option("display.width", 220)
R = lambda n: pd.read_csv(paths.result(n))
print("repo root:", paths.ROOT)

repo root: C:\Users\njari\Desktop\Faks\master


## 1. Response to a single injected row

One row is added to the context with y = dose times the largest training value, at the
centre of feature space. The response is the median relative change of the local
Q(0.99). A median alone would hide the shape of the response, and doing exactly that
produced three separate corrections in this project, so the spread travels with it.

In [2]:
inf = tables.ok_rows(pd.concat([R("influence.csv"), R("influence_exaone.csv")]))
print("%d runs, models %s" % (len(inf), sorted(inf.model.unique())))
print("doses %s, xi %s" % (sorted(inf.dose.unique()), sorted(inf.xi.unique())))
print()
top = inf[inf.dose == inf.dose.max()]
print("at the largest dose (%g x max), local Q(0.99):" % inf.dose.max())
print(tables.median_response(top, "local_q99").round(3).to_string())

180 runs, models ['EXAONE', 'GBM', 'TabICLv2', 'TabPFN-V3']
doses [np.float64(1.0), np.float64(3.0), np.float64(5.0), np.float64(10.0), np.float64(100.0)], xi [np.float64(0.3), np.float64(0.7), np.float64(0.9)]

at the largest dose (100 x max), local Q(0.99):
           median    min    max   n
model                              
TabICLv2   -0.312 -0.614 -0.166  10
TabPFN-V3  -0.300 -0.549  2.612  10
GBM         0.039 -0.032  0.332  10
EXAONE      0.076 -0.023  7.596  15


The control models are the point of the table: gradient boosting and XGBoost do not
standardise the target and do not move, at any dose. Two models are bimodal, so their
median is a summary of a distribution with two peaks and must be reported with the range.

In [3]:
print("dose response, median local Q(0.99) change by model:")
piv = top.pivot_table(index="model", columns="xi", values="local_q99", aggfunc="median")
print(piv.round(3).to_string())
print()
print("negative control, a row at the median instead of the tail:")
if (inf.dose == 1).any():
    print(tables.median_response(inf[inf.dose == 1], "local_q99").round(4).to_string())

dose response, median local Q(0.99) change by model:
xi           0.3    0.7   0.9
model                        
EXAONE     0.007  0.208  0.73
GBM        0.031  0.111   NaN
TabICLv2  -0.234 -0.490   NaN
TabPFN-V3 -0.306 -0.137   NaN

negative control, a row at the median instead of the tail:
           median     min     max   n
model                                
TabPFN-V3  0.0092 -0.0497  0.0299  10
GBM        0.0387 -0.0317  0.3318  10
TabICLv2   0.0786  0.0420  0.1839  10
EXAONE     0.0822  0.0480  0.1421  15


## 2. How often the trigger occurs

The causal quantity is how much the largest point inflates the standard deviation, not
the ratio of the largest to the second largest. Conditioning on the wrong one is what
made an earlier version of this finding conclude there was no effect on real data.

This is measured on 99 public datasets with no model involved at all.

In [4]:
pv = tables.ok_rows(R("prevalence_data.csv"))
print("%d datasets, %d subsamples each" % (len(pv), int(pv.n_subsamples.median())))
print()
for thr in ["p_sd_1.5", "p_sd_2.0", "p_sd_4.0", "p_sd_10.0"]:
    hit = int((pv[thr] > 0).sum())
    print("  sd shift at least %-5s present in %2d of %d datasets  (%.0f%%)"
          % (thr.replace("p_sd_", "") + "x", hit, len(pv), 100 * hit / len(pv)))
print()
worst = pv.nlargest(5, "sd_shift_max")[["dataset", "sd_shift_max", "hill_xi", "p_sd_4.0"]]
print("worst five by the largest sd shift observed:")
print(worst.round(3).to_string(index=False))

99 datasets, 200 subsamples each

  sd shift at least 1.5x  present in  9 of 99 datasets  (9%)
  sd shift at least 2.0x  present in  5 of 99 datasets  (5%)
  sd shift at least 4.0x  present in  2 of 99 datasets  (2%)
  sd shift at least 10.0x present in  1 of 99 datasets  (1%)

worst five by the largest sd shift observed:
                dataset  sd_shift_max  hill_xi  p_sd_4.0
            freMTPL2sev        19.899    0.795     0.110
       Brazilian_houses         6.027    0.302     0.185
   OnlineNewsPopularity         3.337    0.741     0.000
nyc-taxi-green-dec-2016         2.637    0.371     0.000
      Job_Profitability         2.575    0.148     0.000


## 3. Conditioned on that leverage, on real data

The effect appears on the dataset that has both leverage and a heavy tail. On a dataset
with leverage but a lighter tail it does not, which is why this is stated as a
hypothesis resting on two points rather than a law.

In [5]:
pm = tables.ok_rows(R("prevalence_models.csv"))
print("%d runs over %s" % (len(pm), sorted(pm.dataset.unique())))
for ds, g in pm.groupby("dataset"):
    print()
    print(ds)
    print(g.pivot_table(index="bin", columns="model", values="dq99",
                        aggfunc="median").round(3).to_string())

220 runs over ['Brazilian_houses', 'OnlineNewsPopularity', 'freMTPL2sev']

Brazilian_houses
model      GBM  TabICLv2  TabPFN-V3  XGBoost
bin                                         
0.0-1.2 -0.057     0.000      0.000    0.003
1.2-2.0 -0.018     0.002      0.007    0.007
2.0-4.0  0.156     0.007      0.065    0.001
4.0-inf -0.108     0.011      0.076    0.006

OnlineNewsPopularity
model      GBM  TabICLv2  TabPFN-V3  XGBoost
bin                                         
0.0-1.2 -0.050     0.035      0.010    0.029
1.2-2.0 -0.018     0.022     -0.004    0.040
2.0-4.0  0.029    -0.038      0.013    0.121

freMTPL2sev
model      GBM  TabICLv2  TabPFN-V3  XGBoost
bin                                         
0.0-1.2  0.001     0.056      0.045    0.032
1.2-2.0  0.014     0.062     -0.028    0.030
2.0-4.0  0.006    -0.155     -0.017    0.021
4.0-inf  0.000    -0.283      0.039    0.022


## 4. Removing the real largest point

Without any injection, dropping the genuinely largest row moves Q(0.99) only slightly,
because in real heavy-tailed data no single point dominates the standard deviation:
there is a whole upper tail rather than one isolated spike.

In [6]:
ir = tables.ok_rows(R("influence_real.csv"))
print("%d runs over %d datasets" % (len(ir), ir.dataset.nunique()))
print()
print(ir.groupby("model")[["dq50", "dq90", "dq99"]].median().round(4).to_string())
print()
print("leverage present in these samples: median %.2f, max %.2f"
      % (ir.leverage.median(), ir.leverage.max()))
print("Compare with the injected doses above, which reach a far larger sd shift.")

120 runs over 4 datasets

             dq50    dq90    dq99
model                            
GBM        0.0000  0.0044  0.0155
TabICLv2  -0.0029  0.0028  0.0054
TabPFN-V3  0.0001  0.0043  0.0066

leverage present in these samples: median 1.33, max 4.60
Compare with the injected doses above, which reach a far larger sd shift.


## 5. The mechanism, from the output head

For a model whose head is a fixed grid of bins, the inflated standard deviation
stretches the support, so the number of bins covering the actual data range collapses.
That is why its response is bimodal rather than smooth.

In [7]:
for name, lab in [("architecture_tabdpt.csv", "TabDPT, fixed bin grid"),
                  ("architecture_tabfm.csv", "TabFM, mean only")]:
    d = tables.ok_rows(R(name))
    if not len(d):
        print(lab, ": no successful rows"); continue
    cols = [c for c in ["ratio_true", "ratio_truncated", "mean_influence",
                        "bound_coverage"] if c in d.columns]
    print("--- %s  (%d rows)" % (lab, len(d)))
    print(d.groupby("xi")[cols].median().round(3).to_string())
    print()

--- TabDPT, fixed bin grid  (15 rows)
     ratio_true  ratio_truncated  mean_influence  bound_coverage
xi                                                              
0.5       1.005            1.057          -1.294             1.0
0.7       0.919            1.073          -1.880             1.0
0.9       0.473            1.073          -3.071             1.0

--- TabFM, mean only  (6 rows)
     ratio_true  ratio_truncated  mean_influence
xi                                              
0.5       0.959            1.026           2.975
0.9       0.478            1.076          15.722



## 6. The mean, for the models that publish only a mean

In [8]:
me = tables.ok_rows(R("mean_exaone.csv"))
print("EXAONE mean response, %d runs" % len(me))
cols = [c for c in ["ratio_true", "ratio_truncated", "mean_influence"] if c in me.columns]
print(me.groupby(["xi", "dose"])[cols].median().round(3).to_string())

EXAONE mean response, 9 runs
           ratio_true  ratio_truncated  mean_influence
xi  dose                                              
0.5 100.0       0.887            1.018           0.053
0.7 100.0       0.723            0.965           0.229
0.9 100.0       0.385            0.866           0.437


## Disagreements with the text

In [9]:
notes = [
 "the real-data effect rests on one dataset with both leverage and a heavy tail; "
 "run/ carries a driver that widens this to nine datasets before it is called a law.",
 "the largest injected dose is far above anything observed in nature, where the worst "
 "sd shift over 99 datasets is about 20x; the dose response, not the largest dose, is "
 "the honest summary.",
 "two models respond bimodally, so any single number for them is a summary of a "
 "two-peaked distribution and is reported here with its range.",
]
for n in notes:
    print("-", n)

- the real-data effect rests on one dataset with both leverage and a heavy tail; run/ carries a driver that widens this to nine datasets before it is called a law.
- the largest injected dose is far above anything observed in nature, where the worst sd shift over 99 datasets is about 20x; the dose response, not the largest dose, is the honest summary.
- two models respond bimodally, so any single number for them is a summary of a two-peaked distribution and is reported here with its range.
